# Week 3 · Day 3: Decision boundaries and thresholds

**Course:** Lab 03: Decision boundary

The teaching question is **“Is this source's absolute G magnitude below 4?”** Class 1 means yes. This is a brightness label, not a verified stellar type. Only `bp_rp` (color) enters the model; magnitude and parallax are used to construct the answer and must stay out of the inputs.

Day 1 saves a source-disjoint 60/20/20 split. Days 2–6 explore training and validation; Day 7 evaluates the reserved test set. Absolute magnitudes are uncorrected for dust, and the archive sample may not represent all Gaia sources.

## Setup

[Prerequisites and setup explanation](week3_prerequisites.ipynb). Run cells from top to bottom. Run Day 1 first. This notebook reads its saved data.

In [ ]:
from pathlib import Path
import sys
import importlib
import numpy as np
import pandas as pd
from IPython.display import display

# Jupyter may start in the project folder or in its notebooks folder.
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src/classification.py').is_file():
    raise RuntimeError('Open Jupyter from gaia-explorer or gaia-explorer/notebooks.')
sys.path.insert(0, str(ROOT))
from src import classification as data, visualize
# Refresh the data loader if this kernel imported an older version.
data = importlib.reload(data)
visualize = importlib.reload(visualize)  # Pick up updated plot labels and layout.

DATA_PATH = None  # None uses the shared W2 cache; a missing cache is downloaded.
OUTPUT = ROOT / 'data/processed/week3'
OUTPUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 20)


## Load the Day 1 data

These lines select the saved splits. `X` contains color and `y` contains the brightness label.

In [ ]:
prepared_path = OUTPUT / 'learning_data.csv'
if not prepared_path.is_file():
    raise FileNotFoundError('Run Week 3 Day 1 first to create learning_data.csv.')

# Keep source IDs as text so their digits are not rounded.
gaia = pd.read_csv(prepared_path, dtype={'source_id': 'string'})
train = gaia.loc[gaia['split'] == 'train'].copy()
validation = gaia.loc[gaia['split'] == 'validation'].copy()
test = gaia.loc[gaia['split'] == 'test'].copy()
feature_columns = ['bp_rp']
X_train = train[feature_columns]
y_train = train['is_bright']
X_val = validation[feature_columns]
y_val = validation['is_bright']
print('Training rows:', len(train), '| Validation rows:', len(validation))


## 1. Fit a simple baseline

Now scikit-learn learns weights from training data. The pipeline is simply a scaler followed by logistic regression. Expected output: validation probabilities.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# The pipeline learns the scaling and model from training rows only.
model = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=5000))
model.fit(X_train, y_train)
probabilities = model.predict_proba(X_val)[:, 1]


## See what the pipeline calculates

`model.fit(...)` learned a training mean, a training standard deviation, a weight `w`, and an intercept `b`. `model.predict_proba(X_val)` applies these three operations inside the pipeline. The cell below writes them out so you can see where the equations appear. `decision_function` returns the scores before sigmoid; we calculate those scores directly here.

In [ ]:
scaler = model.named_steps['standardscaler']
regression = model.named_steps['logisticregression']

# Step 1: color measured relative to the training colors.
scaled_color = scaler.transform(X_val)

# Step 2: linear score z = w*x + b.
score = scaled_color @ regression.coef_[0] + regression.intercept_[0]

# Step 3: turn each score into the probability of class 1.
probability_from_equation = 1 / (1 + np.exp(-score))

# The pipeline's output and our visible calculation should match.
print('First five scores:', score[:5])
print('First five class-1 probabilities:', probability_from_equation[:5])
print('Matches predict_proba:', np.allclose(probabilities, probability_from_equation))

## 2. View actual labels and model probabilities separately

The **top panel** has one dot per validation star. A dot on row **0** means the star's real label is 0 (`M_G ≥ 4`); a dot on row **1** means its real label is 1 (`M_G < 4`). These are known answers, not probabilities.

The **bottom panel** shows the model's *estimated chance of label 1* for each BP-RP color. The orange curve is a probability between 0% and 100%. The dashed line is the 50% decision threshold: orange above it predicts label 1; orange below it predicts label 0. Where the orange line crosses the dashed line is the color boundary.

Both panels use the same horizontal BP-RP scale. If top-panel dots at both 0 and 1 have similar colors, the model cannot always tell them apart using color alone.

In [ ]:
threshold = 0.5
predictions = (probabilities >= threshold).astype(int)
visualize.plot_color_probability(model, validation, threshold=threshold)

## 3. Find the boundary in color units

At probability threshold `t`, the boundary score is `log(t / (1 - t))`. For `t = 0.5`, that score is **0**. We want to know which original BP-RP color produces that score.

The scaler and logistic model calculate:

```text
scaled_color = (color - training_mean) / training_scale
score = w * scaled_color + b
```

Work backward from the boundary score:

```text
scaled_color = (score_boundary - b) / w
color_boundary = training_mean + training_scale * scaled_color
```

Combining those lines produces the formula in the code cell below. For example, if `training_mean = 1.2`, `training_scale = 0.5`, `w = -1`, and `b = -1`, a score boundary of 0 gives `color_boundary = 1.2 + 0.5 * (0 - (-1)) / (-1) = 0.7`. These numbers illustrate the calculation; the cell uses the actual values learned from the Gaia training stars.

A nearly zero weight would require division by nearly zero and does not give a useful color boundary. The calculated boundary can also fall outside the observed color range.

In [ ]:
scaler = model.named_steps['standardscaler']
regression = model.named_steps['logisticregression']
w = regression.coef_[0, 0]
b = regression.intercept_[0]
if abs(w) < 1e-12:
    print('The weight is nearly zero; there is no useful color boundary.')
else:
    score_boundary = np.log(threshold / (1 - threshold))
    color_boundary = scaler.mean_[0] + scaler.scale_[0] * (score_boundary - b) / w
    print('BP-RP boundary:', color_boundary)
    print('Positive predictions are on the higher-color side if w > 0, otherwise the lower-color side.')

## 4. Compare thresholds

Expected output: precision and recall at 0.3, 0.5, and 0.7. Change the plot threshold to compare. We keep 0.5 for the final experiment, so this exploration does not change the test decision rule.

In [ ]:
from sklearn.metrics import precision_score, recall_score
rows = []
for threshold in [0.3, 0.5, 0.7]:
    predicted = (probabilities >= threshold).astype(int)
    rows.append({
        'threshold': threshold,
        'precision': precision_score(y_val, predicted, zero_division=0),
        'recall': recall_score(y_val, predicted, zero_division=0),
    })
display(pd.DataFrame(rows))

## Reflection questions and answers

1. **Does a threshold change the learned weights?** No. The weights `w` and intercept `b` were learned when `model.fit(X_train, y_train)` ran. Changing the threshold does not fit the model again, so each star keeps the same predicted probability. It only changes whether a probability is turned into label 0 or label 1. For example, probability 0.6 becomes class 1 at threshold 0.5 and class 0 at threshold 0.7.

2. **What happens as the threshold falls?** More stars are classified as class 1 (`abs_g_mag < 4`). This cannot reduce the number of actual class 1 stars found, so recall stays the same or increases. It may also label more class 0 stars as class 1, creating more false positives. Precision may rise or fall depending on which stars are added; inspect the table rather than assuming a direction.

3. **Why is magnitude absent from the boundary calculation?** The classification label is calculated from absolute magnitude: `is_bright = 1` when `abs_g_mag < 4`. If absolute magnitude were also a model input, the model would already have the answer. This model receives only `bp_rp` color, so its decision boundary must be expressed as a BP-RP color value. The HR plot can still show magnitude to help *us* understand the labels.


## References

Course labs: `../ML-coursework/W3/Optional Labs/` in the workspace.

[ESA Gaia DR3](https://www.cosmos.esa.int/web/gaia/dr3) · [Extinction guidance](https://www.cosmos.esa.int/web/gaia/edr3-extinction-law) · [Scikit-learn logistic regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)